# 2 — Construcción de Silver

Silver tipa, deduplica y valida. Los registros aceptados se integran con `MERGE`; los rechazados quedan en cuarentena con una causa explícita. La corrección de la transacción 42 permite observar un `UPDATE` real.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
%run ../common/quality_rules

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
bronze_all = qualified_table(config, "bronze_transactions_all")
assert spark.catalog.tableExists(bronze_all), "Falta bronze_transactions_all. Ejecutá la tarea 01."
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
quarantine = qualified_table(config, "silver_transactions_quarantine")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {customers} USING DELTA AS
SELECT customer_id, country, segment, created_date, email
FROM (
  SELECT try_cast(customer_id AS BIGINT) customer_id, country, segment,
         try_cast(created_date AS DATE) created_date, lower(trim(email)) email,
         row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_customers')}
) WHERE rn = 1 AND customer_id IS NOT NULL
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {products} USING DELTA AS
SELECT product_id, category, price
FROM (
  SELECT try_cast(product_id AS BIGINT) product_id, category, try_cast(price AS DECIMAL(12,2)) price,
         row_number() OVER (PARTITION BY product_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_products')}
) WHERE rn = 1 AND product_id IS NOT NULL AND price > 0
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
raw_columns = ["transaction_id", "customer_id", "product_id", "event_ts", "amount", "payment_channel", "device_id", "is_fraud", "source_batch_id", "updated_at"]
typed = add_transaction_types(spark.table(bronze_all).dropDuplicates(raw_columns))
known_customers = spark.table(customers).select(F.col("customer_id").alias("known_customer"))
known_products = spark.table(products).select(F.col("product_id").alias("known_product"))
checked = (typed
    .join(known_customers, F.col("customer_id_typed") == F.col("known_customer"), "left")
    .join(known_products, F.col("product_id_typed") == F.col("known_product"), "left"))
checked = add_quality_reason(checked)
record_key = F.coalesce(F.col("transaction_id_typed").cast("string"), F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("<NULL>")) for c in raw_columns]), 256))
latest = (checked.withColumn("_record_key", record_key)
    .withColumn("_rn", F.row_number().over(Window.partitionBy("_record_key").orderBy(F.col("updated_at_typed").desc_nulls_last(), F.col("source_batch_id").desc())))
    .where("_rn = 1"))

In [0]:
valid = (latest.where(F.col("quality_reason").isNull()).select(
    F.col("transaction_id_typed").alias("transaction_id"),
    F.col("customer_id_typed").alias("customer_id"),
    F.col("product_id_typed").alias("product_id"),
    F.col("event_ts_typed").alias("event_ts"),
    F.col("amount_typed").alias("amount"), "payment_channel", "device_id",
    F.col("is_fraud_typed").alias("is_fraud"), "source_batch_id",
    F.col("updated_at_typed").alias("updated_at"), F.current_timestamp().alias("processed_at")))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {transactions} (
 transaction_id BIGINT, customer_id BIGINT, product_id BIGINT, event_ts TIMESTAMP, amount DECIMAL(12,2),
 payment_channel STRING, device_id STRING, is_fraud INT, source_batch_id STRING, updated_at TIMESTAMP, processed_at TIMESTAMP
) USING DELTA""")
valid.createOrReplaceTempView("_valid_transactions")
spark.sql(f"""MERGE INTO {transactions} t USING _valid_transactions s ON t.transaction_id = s.transaction_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
rejected = latest.where(F.col("quality_reason").isNotNull()).select(
    *raw_columns, "quality_reason", F.current_timestamp().alias("rejected_at"))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {quarantine} (
 transaction_id STRING, customer_id STRING, product_id STRING, event_ts STRING, amount STRING,
 payment_channel STRING, device_id STRING, is_fraud STRING, source_batch_id STRING, updated_at STRING,
 quality_reason STRING, rejected_at TIMESTAMP
) USING DELTA""")
rejected.createOrReplaceTempView("_rejected_transactions")
spark.sql(f"""MERGE INTO {quarantine} t USING _rejected_transactions s
ON t.transaction_id <=> s.transaction_id AND t.source_batch_id <=> s.source_batch_id AND t.quality_reason = s.quality_reason
WHEN NOT MATCHED THEN INSERT *""")
display(spark.table(quarantine).groupBy("source_batch_id", "quality_reason").count().orderBy("source_batch_id", "quality_reason"))
display(spark.table(transactions).groupBy("source_batch_id").count().orderBy("source_batch_id"))
display(spark.table(quarantine))
display(spark.sql(f"DESCRIBE HISTORY {transactions}"))

source_batch_id,quality_reason,count
batch_002,INVALID_AMOUNT,1
batch_002,UNKNOWN_CUSTOMER,1
batch_003,INVALID_AMOUNT,1
batch_003,UNKNOWN_CUSTOMER,1
initial,INVALID_AMOUNT,51


source_batch_id,count
batch_002,200
batch_003,201
initial,49948


transaction_id,customer_id,product_id,event_ts,amount,payment_channel,device_id,is_fraud,source_batch_id,updated_at,quality_reason,rejected_at
0,2049,497,2026-03-07 02:30:01,N/A,wallet,device_2102,0,initial,2026-03-07 02:30:01,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
10967,778,279,2026-03-02 12:31:22,N/A,card,device_1181,0,initial,2026-03-02 12:31:22,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
11964,1058,240,2026-03-03 13:17:52,N/A,wallet,device_2038,0,initial,2026-03-03 13:17:52,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
12961,2316,476,2026-03-03 16:17:37,N/A,card,device_1854,0,initial,2026-03-03 16:17:37,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
13958,2639,223,2026-03-05 08:42:05,N/A,transfer,device_1335,1,initial,2026-03-05 08:42:05,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
14955,1793,234,2026-03-05 20:50:03,N/A,card,device_2175,0,initial,2026-03-05 20:50:03,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
15952,1625,132,2026-03-03 13:12:23,N/A,transfer,device_924,0,initial,2026-03-03 13:12:23,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
16949,267,343,2026-03-04 12:24:01,N/A,wallet,device_900,0,initial,2026-03-04 12:24:01,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
17946,847,343,2026-03-05 06:24:40,N/A,transfer,device_807,0,initial,2026-03-05 06:24:40,INVALID_AMOUNT,2026-10-03T20:33:33.608Z
18943,3578,57,2026-03-03 17:57:52,N/A,wallet,device_1997,0,initial,2026-03-03 17:57:52,INVALID_AMOUNT,2026-10-03T20:33:33.608Z


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-10-07T18:36:30.000Z,76500955381150,jmena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#10929L = transaction_id#10907L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#10913 > updated_at#10938)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(377354366942216),299dfbc2-8fc0-477b-9f8e-3095721a3547,1007-183516-i6189mky-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 11265, materializeSourceTimeMs -> 2889, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 5870, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50349, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2328)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
5,2026-10-05T19:38:48.000Z,76500955381150,jmena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#11900L = transaction_id#11878L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#11884 > updated_at#11909)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(377354366942216),bd104acb-2917-4c1d-9a46-6b8cf60d0d6a,1005-192827-87ce26vz-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 8348, materializeSourceTimeMs -> 2273, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3827, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50349, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2103)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-05T19:08:30.000Z,76500955381150,jmena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#11545L = transaction_id#11523L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#11529 > updated_at#11554)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(929729170742187, bigdata_jmena_silver_gold, 427588968054164, 510836159997637, 76500955381150, manual, 7474648327291677)",List(377354366942216),570f1ae9-83a3-42b2-9853-7efb9df012c1,1005-190704-rcpll1nh-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 8990, materializeSourceTimeMs -> 3056, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3815, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50349, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1981)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-03T21:37:28.000Z,765

In [0]:
display(spark.table(transactions).where("transaction_id = 42"))

transaction_id,customer_id,product_id,event_ts,amount,payment_channel,device_id,is_fraud,source_batch_id,updated_at,processed_at
42,0,0,2026-03-11T00:00:00.000Z,1999.99,card,device_corrected_42,1,batch_003,2026-03-11T02:00:00.000Z,2026-10-03T21:37:16.436Z


In [0]:
duplicados = (spark.table(transactions)
              .groupBy("transaction_id")
              .count()
              .filter(F.col("count") > 1)
              .orderBy(F.col("count").desc())
)
display(duplicados)

transaction_id,count


In [0]:
display(spark.table(transactions))

transaction_id,customer_id,product_id,event_ts,amount,payment_channel,device_id,is_fraud,source_batch_id,updated_at,processed_at
1,3190,412,2026-03-02T05:17:39.000Z,408.25,wallet,device_1049,0,initial,2026-03-02T05:17:39.000Z,2026-10-03T20:33:22.757Z
10,45,244,2026-03-02T08:25:39.000Z,526.12,wallet,device_1607,0,initial,2026-03-02T08:25:39.000Z,2026-10-03T20:33:22.757Z
100,492,49,2026-03-03T08:07:00.000Z,1707.43,card,device_154,0,initial,2026-03-03T08:07:00.000Z,2026-10-03T20:33:22.757Z
1000,3037,346,2026-03-01T05:57:15.000Z,971.66,transfer,device_1415,0,initial,2026-03-01T05:57:15.000Z,2026-10-03T20:33:22.757Z
10000,2107,439,2026-03-06T09:45:27.000Z,1167.70,wallet,device_1586,0,initial,2026-03-06T09:45:27.000Z,2026-10-03T20:33:22.757Z
10001,2644,160,2026-03-05T23:52:54.000Z,1097.15,wallet,device_749,0,initial,2026-03-05T23:52:54.000Z,2026-10-03T20:33:22.757Z
10002,3264,376,2026-03-07T08:53:49.000Z,979.14,wallet,device_2277,0,initial,2026-03-07T08:53:49.000Z,2026-10-03T20:33:22.757Z
10003,841,473,2026-03-06T17:59:44.000Z,102.72,card,device_687,0,initial,2026-03-06T17:59:44.000Z,2026-10-03T20:33:22.757Z
10004,2786,315,2026-03-07T04:28:01.000Z,527.22,card,device_1123,0,initial,2026-03-07T04:28:01.000Z,2026-10-03T20:33:22.757Z
10005,1835,57,2026-03-01T18:01:55.000Z,653.89,card,device_1377,0,initial,2026-03-01T18:01:55.000Z,2026-10-03T20:33:22.757Z
